# Fase 2 - Coleta de Dados
**Projeto:** Auditoria de Dados · Portal da Transparência Federal
**Notebook:** coleta.ipynb
**Objetivo:** Carregar os arquivos CSV de viagens baixados manualmente do Portal da Transparência, tratar os registros e salvar os arquivos para as fases seguintes.
**Fonte:** [Portal da Transparência - Download de Dados](https://portaldatransparencia.gov.br/download-de-dados/viagens)
**Período:** Janeiro/2022 - Dezembro/2024

> **Nota metodológica:** A coleta via API foi substituída por download direto dos CSVs disponibilizados pelo Portal da Transparência. O motivo é a presença de WAF que bloqueia requisições automatizados com erro 403. Os dados são idênticos - apenas método de acesso difere. Isso esta documentado no arquivo de linhagem (`data/processed/linhagem.json`)

---

## Como obter os arquivos

Antes de executar este notebook, baixe os arquivos em:
```
https://portaldatransparencia.gov.br/download-de-dados/viagens
```
Selecione os anos **2022, 2023 e 2024**, baixe e descompacte cada arquivo em `data/raw`. Renomeie para o padrão:
`viagens_2022.csv`
`viagens_2023.csv`
`viagens_2024.csv`

---
## Sumário
1. [Configuração do ambiente](#1-configuração-do-ambiente)
2. [Carregamento dos arquivos locais](#2-carregamento-dos-arquivos-locais)
3. [Inspeção inicial dos dados](#3-inspeção-inicial-dos-dados)
4. [Mapeamento das colunas reais](#4-mapeamento-das-colunas-reais)
5. [Tratamento e padronização](#5-tratamento-e-padronização)
6. [Separação: diárias e passagens](#6-separação-diárias-e-passagens)
7. [Documentação de linhagem](#7-documentação-de-linhagem)
8. [Salvamento dos arquivos processados](#8-salvamento-dos-arquivos-processados)
9. [Resumo da coleta](#9-resumo-da-coleta)

---
## 1.Configuração do Ambiente

In [6]:
import pandas as pd
import numpy as np
import os
import glob
import json
from datetime import datetime

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Ambiente preparado. Pronto para processar os dados.')
print(f'Data de início: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


Ambiente preparado. Pronto para processar os dados.
Data de início: 2026-05-17 16:50:39


---
## 2. Carregamento dos Arquivos Locais

Lemos todos os arquivos `viagens_*.csv` da pasta `data/raw/` e os concatenamos em um único DataFrame.

Por que `dtype=str`?  
Carregamos tudo como texto na primeira leitura para evitar que o pandas interprete CPFs como números (perdendo zeros à esquerda) ou converta datas de forma. A conversão de tipos acontece de forma controlada na Seção 5.

In [7]:
def carregar_viagens_local(pasta: str) -> pd.DataFrame:
    arquivos = sorted(glob.glob(os.path.join(pasta, 'viagens_*.csv')))

    if not arquivos:
        raise FileNotFoundError(f'Nenhum arquivo encontrado em {pasta}')
    
    frames = []
    for arq in arquivos:
        print(f'Carregando {os.path.basename(arq)}...', end=' ')
        df = pd.read_csv(
            arq,
            sep=';',
            encoding='latin-1',
            dtype = str,
            on_bad_lines='skip'
        )
        df['arquivo_origem']= os.path.basename(arq)
        frames.append(df)
        print(f'{len(df):,} registros carregados.')
    
    df_concatenado = pd.concat(frames, ignore_index=True)
    print(f'\n Total carregado: {len(df_concatenado):,} registros | {df_concatenado.shape[1]} colunas.')  
    return df_concatenado 

df_raw = carregar_viagens_local('../data/raw')
    

Carregando viagens_2022.csv... 662,208 registros carregados.
Carregando viagens_2023.csv... 833,135 registros carregados.
Carregando viagens_2024.csv... 799,441 registros carregados.

 Total carregado: 2,294,784 registros | 23 colunas.


In [8]:
df_raw.to_csv('../data/raw/viagens_raw_2022_2024.csv', index=False)
print('Arquivo concatenado salvo em ../data/raw/viagens_raw_2022_2024.csv')

Arquivo concatenado salvo em ../data/raw/viagens_raw_2022_2024.csv


---
## 3. Inspeção Inicial dos Dados

Antes de qualquer tratamento, documentamos o estado bruto: colunas disponíveis, tipos, nulos e amostra.

**Esta etapa é obrigatória** -- os nomes de colunas do portal podem variar entre versões e precisam ser conferidos antes de prosseguir para o mapemaneto da Seção 4.

In [4]:
print(f'Dimensões : {df_raw.shape[0]:,} registros x {df_raw.shape[1]} colunas')
print(f'Memória usada: {df_raw.memory_usage(deep=True).sum() / (1024**2):.2f} MB')
print()
print('COLUNAS DISPONÍVEIS (use esta lista para ajustar o RENAME_MAP no Seção 4):')
print('-'*65)
for i, col in enumerate(df_raw.columns):
    nulos = df_raw[col].isna().sum()
    pct = nulos / len(df_raw) * 100
    print(f' {i:>2}. {col:<45} nulos: {nulos:>8,} ({pct:.1f})%')

Dimensões : 4,240,256 registros x 24 colunas
Memória usada: 3176.77 MB

COLUNAS DISPONÍVEIS (use esta lista para ajustar o RENAME_MAP no Seção 4):
-----------------------------------------------------------------
  0. Identificador do processo de viagem           nulos: 1,945,472 (45.9)%
  1. Número da Proposta (PCDP)                     nulos: 1,945,472 (45.9)%
  2. Situação                                      nulos: 1,945,472 (45.9)%
  3. Viagem Urgente                                nulos: 1,945,472 (45.9)%
  4. Justificativa Urgência Viagem                 nulos: 1,945,820 (45.9)%
  5. Código do órgão superior                      nulos: 1,945,472 (45.9)%
  6. Nome do órgão superior                        nulos: 1,945,472 (45.9)%
  7. Código órgão solicitante                      nulos: 1,945,472 (45.9)%
  8. Nome órgão solicitante                        nulos: 1,945,472 (45.9)%
  9. CPF viajante                                  nulos: 1,952,952 (46.1)%
 10. Nome                  